# Limpieza del dataset

Este notebook corresponde a la issue #7. Aplica las reglas acordadas para convertir `price`, tratar valores inválidos y conservar los registros válidos. Los outliers positivos de precio no se eliminan porque pueden representar alojamientos premium.

Los nulos de las variables predictoras se mantienen para que el pipeline posterior los impute usando únicamente el train.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path("..").resolve()))
from src.cleaning import clean_listings

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.2)
DATA_PATH = Path("../data/raw/listings.csv.gz")
df = pd.read_csv(DATA_PATH, compression="gzip")
cleaned = clean_listings(df)
print(f"Filas originales: {len(df):,}")
print(f"Filas tras exigir price válido: {len(cleaned):,}")
print(f"Filas eliminadas por price inválido/nulo: {len(df) - len(cleaned):,}")

Filas originales: 15,236
Filas tras exigir price válido: 12,610
Filas eliminadas por price inválido/nulo: 2,626


## Conversión de `price` y valores inválidos

In [2]:
display(cleaned[["price"]].describe().round(2))
print("Precios nulos tras la limpieza:", cleaned["price"].isna().sum())
print("Precios no positivos tras la limpieza:", (cleaned["price"] <= 0).sum())

q1, q3 = cleaned["price"].quantile([0.25, 0.75])
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
price_outliers = cleaned["price"] > upper_bound
print(f"Umbral IQR superior: €{upper_bound:,.2f}")
print(f"Outliers IQR detectados: {price_outliers.sum():,} ({price_outliers.mean():.2%})")
print("Los outliers se conservan por ser precios positivos potencialmente válidos.")

,price
count,12610.0
mean,253.41
std,386.34
min,12.81
25%,93.38
50%,190.67
75%,284.0
max,11245.03


Precios nulos tras la limpieza: 0
Precios no positivos tras la limpieza: 0
Umbral IQR superior: €569.92
Outliers IQR detectados: 635 (5.04%)
Los outliers se conservan por ser precios positivos potencialmente válidos.


## Imputaciones con significado de negocio

In [3]:
summary = pd.DataFrame({
    "missing_before": df[["beds", "minimum_nights", "reviews_per_month"]].isna().sum(),
    "missing_after": cleaned[["beds", "minimum_nights", "reviews_per_month"]].isna().sum(),
})
display(summary)

print("beds imputados:", cleaned["beds_was_missing"].sum())
print("minimum_nights pendientes para imputación train-only:", cleaned["minimum_nights_was_missing"].sum())
print("reviews_per_month imputados a cero:", ((cleaned["has_reviews"] == 0) & (cleaned["reviews_per_month"] == 0)).sum())

display(cleaned.groupby("room_type", dropna=False)[["beds", "minimum_nights", "reviews_per_month"]].median().round(2))

,missing_before,missing_after
beds,414,0
minimum_nights,2045,1876
reviews_per_month,3301,0


beds imputados: 349
minimum_nights pendientes para imputación train-only: 1876
reviews_per_month imputados a cero: 2486


,beds,minimum_nights,reviews_per_month
room_type,,,
Entire home/apt,3.0,2.0,1.00
Hotel room,1.0,1.0,0.71
Private room,1.0,31.0,0.17
Shared room,4.0,1.0,1.18


## Decisiones

- `price` se convierte a numérico y se eliminan filas sin un precio válido porque no pueden recibir un target fiable.
- Los precios extremos positivos se conservan y solo se marcan para su análisis.
- `beds` se completa con `bedrooms`; los estudios reciben una cama y se conserva un indicador de imputación.
- `minimum_nights` conserva sus nulos: la mediana se aprenderá únicamente con train dentro del pipeline de preprocessing.
- `reviews_per_month` se completa con cero cuando `number_of_reviews` es cero; las puntuaciones y fechas de reseñas permanecen nulas.
- `has_reviews` y los indicadores de imputación preservan información sobre los valores ausentes.